# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Lane: Refresh / Content Opportunity Scoring

I frame this lane primarily as a **ranking / scoring task**. The goal is to assign each content page a priority score and rank pages according to which ones should receive human review first.

This is more useful than simply classifying a page as declining or not declining because the content team has limited review capacity. A ranked output helps the team focus first on higher-priority pages and then decide whether to refresh, expand, improve, monitor, or leave them unchanged.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / Proxy

For this ranking task, the intended target is a **review-priority signal** that helps determine which content pages should receive human attention first.

The starter dataset does not contain a direct observed label for whether a page should be refreshed. It does contain `trend_direction` and `trend_pct`, which describe recent performance trends. These can be used as **exploratory proxies** for identifying potentially declining content, but they should not be treated as proof that refreshing a page will improve its performance.

For later modeling, I would prefer a target based on an **observed outcome in a future time window**, such as whether a page experiences a meaningful performance decline after the feature observation period. This would allow historical signals to be used to predict a later observed outcome rather than defining the target from the same information used as features.

For now, the target remains provisional because this assignment focuses on framing the ML task rather than training the final model.

In [2]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/Viditp25/flyrank-intern-/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)

# Inspect possible proxy columns in the starter dataset
print("Trend direction distribution:")
print(df["trend_direction"].value_counts())

print("\nTrend percentage summary:")
print(df["trend_pct"].describe())

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric: Precision@K

The primary success metric for this ranking task would be **Precision@K**, where K represents the number of pages that the content team has capacity to review.

Precision@K measures the proportion of pages in the top-K recommendations that are genuinely relevant review candidates. This is appropriate because the practical goal is not to classify every page correctly, but to make the limited set of pages reviewed first as useful as possible.

For example, if the team can review 100 pages, Precision@100 would measure how many of those 100 recommended pages match the chosen observed outcome or validated review criterion.

A useful ML approach should outperform a transparent rule-based baseline on Precision@K. Therefore, I would define "good" as achieving a meaningfully higher Precision@K than the baseline on held-out data. I would not set an arbitrary percentage before establishing the baseline and a reliable target.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis

The unit of analysis is **one content item (one webpage)**. Each row represents one pseudonymized content item that could potentially be prioritized for human review.

For the Refresh / Content Opportunity Scoring lane, I focus on columns that describe the page's content characteristics, search visibility, engagement, freshness, and recent performance. The `content_id` identifies the content item, while `client_id` identifies the pseudonymized client. These IDs are useful for identification, grouping, and data splitting, but they should not be used as predictive features.

The dataframe below shows what one row of the ranking problem looks like in practice.

In [6]:
lane_columns = [
    "content_id",
    "content_type",
    "main_intent",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "trend_direction"  # exploratory proxy/context only
]

lane_preview = df[lane_columns].copy()

print(f"Lane dataframe shape: {lane_preview.shape}")
print("Unit of analysis: one row = one content item")

display(lane_preview.head())

Lane dataframe shape: (30000, 10)
Unit of analysis: one row = one content item


,content_id,content_type,main_intent,impressions_90d,sessions_90d,avg_position,content_age_days,days_since_last_update,engagement_rate,trend_direction
0,content_304f48230142,keyword article,transactional,3803,17,10.6,187,20,5.88,down
1,content_a1fb4e703a9e,keyword article,informational,15320,9,20.3,445,25,0.00,down
2,content_9aa793d4d895,keyword article,informational,12581,11,36.5,141,20,0.00,down
3,content_331d6c4de07b,keyword article,commercial,11751,78,6.2,463,22,1.28,stable
4,content_d99b7a2d90ca,keyword article,informational,19140,145,44.0,263,14,0.00,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats a Fixed Rule Here

A fixed rule could provide a useful baseline. For example, pages could be prioritized using simple thresholds based on impressions, engagement, freshness, or recent performance.

However, review priority may depend on interactions among several signals. A page with declining performance but very low visibility may not deserve the same priority as a declining page with high visibility. Similarly, content age, search position, engagement, and time since the last update may affect how useful a review would be.

Machine learning is worth testing because it can potentially learn these interactions across multiple signals instead of relying on a small set of manually chosen thresholds. The goal is not simply to train a model, but to determine whether a learned ranking can provide better decision support than a transparent rule-based baseline.

If a simple fixed rule performs as well as the ML approach, the simpler rule may be preferable.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.